# 07. Объединение кандидатов

ALS и Content-Based загружаются из Parquet. Validation подробно показывает History, Popularity, flags, concat и groupby; train/test не спрятаны в цикл.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas, метрики и технические helpers только для повторения  
**Зачем:** модели и encoder здесь не импортируются  
**Что получим:** минимальный набор для candidate merge

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

from fashion_recommender.baselines import popular_items
from fashion_recommender.candidates import merge_candidate_sources, popularity_candidates
from fashion_recommender.data import load_transactions
from fashion_recommender.evaluation import (
    candidate_recall_at_k, hit_rate_at_k, map_at_k, mean_recall_at_k,
)
from fashion_recommender.persistence import load_json

### Пути и лимиты

**Что делаем:** задаём входные файлы и размеры источников  
**Зачем:** лимиты должны быть видны до объединения  
**Что получим:** пути и пять констант

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
if not WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл: {WINDOWS_PATH}\n"
        "Сначала выполните notebook 03_temporal_validation_colab.ipynb."
    )

ALS_LIMIT = 150
CONTENT_LIMIT = 50
PERSONAL_LIMIT = 20
POPULARITY_LIMIT = 30
FINAL_LIMIT = 250
MAX_EVALUATION_USERS = 2_000
RANDOM_STATE = 42

### Проверка candidate artifacts

**Что делаем:** проверяем шесть файлов от notebooks 05 и 06  
**Зачем:** отсутствие файла не должно запускать повторное обучение  
**Что получим:** понятную ошибку с producer notebook

In [5]:
required_candidate_paths = [
    PROCESSED_DIR / "als_candidates_train.parquet",
    PROCESSED_DIR / "als_candidates_validation.parquet",
    PROCESSED_DIR / "als_candidates_test.parquet",
    PROCESSED_DIR / "content_candidates_train.parquet",
    PROCESSED_DIR / "content_candidates_validation.parquet",
    PROCESSED_DIR / "content_candidates_test.parquet",
]
missing_candidate_paths = [
    path for path in required_candidate_paths if not path.is_file()
]
if missing_candidate_paths:
    raise FileNotFoundError(
        f"Не найдены candidate artifacts: {missing_candidate_paths}. "
        "Сначала выполните notebooks 05 и 06."
    )

### Загрузка общих входов

**Что делаем:** читаем transactions и temporal windows один раз  
**Зачем:** Personal History и Popularity используют history каждого окна  
**Что получим:** `transactions` и `windows`

In [6]:
transactions = load_transactions(TRANSACTIONS_PATH)
windows = load_json(WINDOWS_PATH)
print("Transactions:", transactions.shape)

Transactions: (1048575, 5)


### Validation history и target

**Что делаем:** выбираем одно окно для подробного merge  
**Зачем:** никакая модель здесь не обучается  
**Что получим:** две таблицы

In [7]:
validation_cutoff = pd.Timestamp(windows["validation"]["cutoff_date"])
validation_end = pd.Timestamp(windows["validation"]["target_end_date"])
validation_history = transactions[
    transactions["t_dat"] < validation_cutoff
].copy()
validation_target = transactions[
    transactions["t_dat"].between(validation_cutoff, validation_end)
].copy()
assert validation_history["t_dat"].max() < validation_cutoff
print("History:", validation_history.shape, "Target:", validation_target.shape)

History: (1011880, 5) Target: (23405, 5)


### Validation ground truth

**Что делаем:** оставляем известных users и уникальные future items  
**Зачем:** source coverage измеряется на полном target  
**Что получим:** полный словарь ответов

In [8]:
validation_known_users = set(validation_history["customer_id"])
validation_target_evaluation = validation_target[
    validation_target["customer_id"].isin(validation_known_users)
]
validation_target_unique = (
    validation_target_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
validation_ground_truth = validation_target_unique.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
print("Ground-truth users:", len(validation_ground_truth))

Ground-truth users: 13078


### Validation cohort

**Что делаем:** повторяем deterministic sample  
**Зачем:** загруженные ALS/Content files построены на тех же users  
**Что получим:** `validation_users` и sample ground truth

In [9]:
validation_all_users = np.array(sorted(validation_ground_truth))
validation_sample_size = min(MAX_EVALUATION_USERS, len(validation_all_users))
validation_rng = np.random.default_rng(RANDOM_STATE)
validation_users = validation_rng.choice(
    validation_all_users,
    size=validation_sample_size,
    replace=False,
).tolist()
validation_ground_truth_sample = {
    customer_id: validation_ground_truth[customer_id]
    for customer_id in validation_users
}
print("Evaluation users:", len(validation_users))

Evaluation users: 2000


### Загрузка ALS validation

**Что делаем:** читаем готовые ALS candidates  
**Зачем:** никакого `AlternatingLeastSquares.fit` здесь нет  
**Что получим:** `validation_als_candidates`

In [10]:
validation_als_path = PROCESSED_DIR / "als_candidates_validation.parquet"
validation_als_candidates = pd.read_parquet(validation_als_path)
display(validation_als_candidates.head())

                                         customer_id  ... als_rank
0  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...        1
1  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...        2
2  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...        3
3  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...        4
4  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...        5

[5 rows x 4 columns]


### Проверка ALS columns

**Что делаем:** проверяем contract и оставляем Top-150  
**Зачем:** notebook 05 сохраняет до 200 для анализа recall curve  
**Что получим:** корректную ALS source table

In [11]:
required_als_columns = {
    "customer_id", "article_id", "als_score", "als_rank"
}
assert required_als_columns <= set(validation_als_candidates.columns)
validation_als_candidates = validation_als_candidates[
    validation_als_candidates["als_rank"] <= ALS_LIMIT
].copy()
print("ALS rows:", len(validation_als_candidates))

ALS rows: 300000


### Загрузка Content validation

**Что делаем:** читаем готовые cosine candidates  
**Зачем:** OneHotEncoder здесь не создаётся  
**Что получим:** `validation_content_candidates`

In [12]:
validation_content_path = PROCESSED_DIR / "content_candidates_validation.parquet"
validation_content_candidates = pd.read_parquet(validation_content_path)
display(validation_content_candidates.head())

                                         customer_id  ... content_rank
0  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...            1
1  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...            2
2  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...            3
3  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...            4
4  f3d49d95bd69cc881ab26f3adb99c47ada2ce324d2773c...  ...            5

[5 rows x 4 columns]


### Проверка Content columns

**Что делаем:** проверяем score и rank  
**Зачем:** ошибка схемы должна быть обнаружена до concat  
**Что получим:** корректную Content source table

In [13]:
required_content_columns = {
    "customer_id", "article_id", "content_similarity_score", "content_rank"
}
assert required_content_columns <= set(validation_content_candidates.columns)
assert validation_content_candidates["content_rank"].max() <= CONTENT_LIMIT
print("Content rows:", len(validation_content_candidates))

Content rows: 100000


### Recent History candidates

**Что делаем:** сортируем history и удаляем повторные пары  
**Зачем:** последние покупки дают один персональный источник  
**Что получим:** Top-20 recent pairs

In [14]:
validation_cohort_history = validation_history[
    validation_history["customer_id"].isin(validation_users)
].copy()
validation_recent = (
    validation_cohort_history
    .sort_values("t_dat", ascending=False)
    .drop_duplicates(["customer_id", "article_id"])
)
validation_recent["personal_history_rank"] = (
    validation_recent.groupby("customer_id").cumcount() + 1
)
validation_recent = validation_recent[
    validation_recent["personal_history_rank"] <= PERSONAL_LIMIT
].copy()
validation_recent["personal_history_score"] = 1 / validation_recent["personal_history_rank"]
display(validation_recent.head())

            t_dat  ... personal_history_score
687272 2019-12-17  ...                    1.0
242574 2019-12-17  ...                    1.0
739069 2019-12-17  ...                    1.0
377949 2019-12-17  ...                    1.0
855144 2019-12-17  ...                    1.0

[5 rows x 7 columns]


### Frequent History candidates

**Что делаем:** считаем count и last purchase каждой пары  
**Зачем:** частота даёт второй персональный порядок  
**Что получим:** Top-20 frequent pairs

In [15]:
validation_frequent = validation_cohort_history.groupby(
    ["customer_id", "article_id"], as_index=False
).agg(
    purchase_count=("article_id", "size"),
    last_purchase=("t_dat", "max"),
)
validation_frequent = validation_frequent.sort_values(
    ["customer_id", "purchase_count", "last_purchase", "article_id"],
    ascending=[True, False, False, True],
)
validation_frequent["personal_history_rank"] = (
    validation_frequent.groupby("customer_id").cumcount() + 1
)
validation_frequent = validation_frequent[
    validation_frequent["personal_history_rank"] <= PERSONAL_LIMIT
].copy()
validation_frequent["personal_history_score"] = validation_frequent["purchase_count"]
display(validation_frequent.head())

                                         customer_id  ... personal_history_score
5  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1
7  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1
0  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1
4  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1
3  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1

[5 rows x 6 columns]


### Общий Personal History

**Что делаем:** складываем recent и frequent pairs, затем удаляем дубли  
**Зачем:** оба baseline-сигнала сохраняются в одном source  
**Что получим:** одну строку на персональную пару

In [16]:
validation_personal_parts = [
    validation_recent[
        ["customer_id", "article_id", "personal_history_score", "personal_history_rank"]
    ],
    validation_frequent[
        ["customer_id", "article_id", "personal_history_score", "personal_history_rank"]
    ],
]
validation_personal_candidates = pd.concat(
    validation_personal_parts,
    ignore_index=True,
)
validation_personal_candidates = validation_personal_candidates.groupby(
    ["customer_id", "article_id"], as_index=False
).agg(
    personal_history_score=("personal_history_score", "max"),
    personal_history_rank=("personal_history_rank", "min"),
)
display(validation_personal_candidates.head())

                                         customer_id  ... personal_history_rank
0  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                     3
1  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                     8
2  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                     7
3  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                     5
4  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                     3

[5 rows x 4 columns]


### Popularity table

**Что делаем:** считаем глобальные counts и ranks  
**Зачем:** source использует только validation history  
**Что получим:** Top-30 товаров

In [17]:
validation_popular_table = popular_items(
    validation_history,
    limit=POPULARITY_LIMIT,
)
display(validation_popular_table.head())

   article_id  popularity_score  popularity_rank
0  0706016001              1743                1
1  0706016002              1427                2
2  0372860001               952                3
3  0706016003               796                4
4  0464297007               787                5


### Popularity candidates

**Что делаем:** повторяем небольшой Top-30 для каждого evaluation user  
**Зачем:** полного user × catalog произведения нет  
**Что получим:** 60 000 строк для cohort 2 000

In [18]:
validation_users_table = pd.DataFrame({
    "customer_id": validation_users,
    "_key": 1,
})
validation_popularity_candidates = validation_users_table.merge(
    validation_popular_table.assign(_key=1),
    on="_key",
).drop(columns="_key")
print("Popularity rows:", len(validation_popularity_candidates))

Popularity rows: 60000


### Флаги источников

**Что делаем:** добавляем по одному бинарному флагу к каждой таблице  
**Зачем:** после concat будет видно происхождение пары  
**Что получим:** четыре source frames

In [19]:
validation_als_source = validation_als_candidates.assign(from_als=1)
validation_content_source = validation_content_candidates.assign(
    from_content_based=1
)
validation_personal_source = validation_personal_candidates.assign(
    from_personal_history=1
)
validation_popularity_source = validation_popularity_candidates.assign(
    from_popularity=1
)

### Concat источников

**Что делаем:** складываем четыре таблицы вертикально  
**Зачем:** до groupby одна пара может встретиться несколько раз  
**Что получим:** `validation_all_candidates`

In [20]:
validation_all_candidates = pd.concat(
    [
        validation_als_source,
        validation_content_source,
        validation_personal_source,
        validation_popularity_source,
    ],
    ignore_index=True,
    sort=False,
)
print("Строк до groupby:", len(validation_all_candidates))

Строк до groupby: 466883


### Одна строка на пару

**Что делаем:** агрегируем scores, ranks и flags  
**Зачем:** одинаковая user-item pair не должна дублироваться  
**Что получим:** `validation_merged_candidates`

In [21]:
validation_merged_candidates = validation_all_candidates.groupby(
    ["customer_id", "article_id"], as_index=False
).agg(
    als_score=("als_score", "max"),
    als_rank=("als_rank", "min"),
    content_similarity_score=("content_similarity_score", "max"),
    content_rank=("content_rank", "min"),
    personal_history_score=("personal_history_score", "max"),
    personal_history_rank=("personal_history_rank", "min"),
    popularity_score=("popularity_score", "max"),
    popularity_rank=("popularity_rank", "min"),
    from_als=("from_als", "max"),
    from_content_based=("from_content_based", "max"),
    from_personal_history=("from_personal_history", "max"),
    from_popularity=("from_popularity", "max"),
)
print("Уникальных пар:", len(validation_merged_candidates))

Уникальных пар: 444853


### Заполнение scores

**Что делаем:** заменяем отсутствующий score нулём  
**Зачем:** NaN означает, что source не предложил пару  
**Что получим:** четыре числовых score columns

In [22]:
score_columns = [
    "als_score",
    "content_similarity_score",
    "personal_history_score",
    "popularity_score",
]
validation_merged_candidates[score_columns] = (
    validation_merged_candidates[score_columns].fillna(0.0)
)
display(validation_merged_candidates[score_columns].head())

   als_score  ...  popularity_score
0   0.010998  ...               0.0
1   0.000000  ...             690.0
2   0.000000  ...             593.0
3   0.010537  ...               0.0
4   0.013579  ...               0.0

[5 rows x 4 columns]


### Заполнение ranks и flags

**Что делаем:** заменяем отсутствующие ranks/flags нулём и задаём типы  
**Зачем:** значение 0 явно означает отсутствие source  
**Что получим:** готовые rank и flag columns

In [23]:
rank_columns = [
    "als_rank", "content_rank", "personal_history_rank", "popularity_rank",
]
flag_columns = [
    "from_als", "from_content_based", "from_personal_history", "from_popularity",
]
validation_merged_candidates[rank_columns] = (
    validation_merged_candidates[rank_columns].fillna(0).astype("int32")
)
validation_merged_candidates[flag_columns] = (
    validation_merged_candidates[flag_columns].fillna(0).astype("int8")
)

### Число источников

**Что делаем:** суммируем четыре бинарных флага  
**Зачем:** пары из нескольких sources получают отдельный признак  
**Что получим:** `number_of_candidate_sources`

In [24]:
validation_merged_candidates["number_of_candidate_sources"] = (
    validation_merged_candidates[flag_columns].sum(axis=1)
)
display(validation_merged_candidates[
    ["customer_id", "article_id", "number_of_candidate_sources"]
].head())

                                         customer_id  ... number_of_candidate_sources
0  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                           1
1  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                           1
2  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                           1
3  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                           1
4  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                           1

[5 rows x 3 columns]


### Ограничение 250

**Что делаем:** сортируем по source count и reciprocal ranks  
**Зачем:** ranking table остаётся ограниченного размера  
**Что получим:** до 250 pairs на пользователя

In [25]:
validation_rank_priority = sum(
    np.where(validation_merged_candidates[column] > 0,
             1 / validation_merged_candidates[column], 0)
    for column in rank_columns
)
validation_merged_candidates["_rank_priority"] = validation_rank_priority
validation_merged_candidates = validation_merged_candidates.sort_values(
    ["customer_id", "number_of_candidate_sources", "_rank_priority"],
    ascending=[True, False, False],
)
validation_merged_candidates = validation_merged_candidates[
    validation_merged_candidates.groupby("customer_id").cumcount() < FINAL_LIMIT
].drop(columns="_rank_priority").reset_index(drop=True)

### Проверки merged table

**Что делаем:** смотрим дубликаты и среднее число candidates  
**Зачем:** одна пара должна встречаться один раз  
**Что получим:** две контрольные статистики

In [26]:
validation_duplicate_pairs = validation_merged_candidates.duplicated(
    ["customer_id", "article_id"]
).sum()
validation_average_candidates = validation_merged_candidates.groupby(
    "customer_id"
).size().mean()
print("Duplicate pairs:", validation_duplicate_pairs)
print("Average candidates:", validation_average_candidates)
assert validation_duplicate_pairs == 0

Duplicate pairs: 0
Average candidates: 222.4265


### Validation Candidate Recall

**Что делаем:** оцениваем объединённое покрытие  
**Зачем:** метрика рассчитана после merge и до ranking  
**Что получим:** Candidate Recall@250

In [27]:
validation_candidate_lists = validation_merged_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
validation_candidate_recall = candidate_recall_at_k(
    validation_ground_truth_sample,
    validation_candidate_lists,
    FINAL_LIMIT,
)
print(f"Candidate Recall@{FINAL_LIMIT}:", validation_candidate_recall)

Candidate Recall@250: 0.05429166666666666


### Вклад validation sources

**Что делаем:** считаем coverage каждого source отдельно  
**Зачем:** видим, какой генератор добавляет future items  
**Что получим:** таблицу из четырёх строк

In [28]:
validation_source_analysis = pd.DataFrame([
    {
        "source": "ALS",
        "pairs": len(validation_als_candidates),
        "candidate_recall": candidate_recall_at_k(
            validation_ground_truth_sample,
            validation_als_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(),
            ALS_LIMIT,
        ),
    },
    {
        "source": "Content-Based",
        "pairs": len(validation_content_candidates),
        "candidate_recall": candidate_recall_at_k(
            validation_ground_truth_sample,
            validation_content_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(),
            CONTENT_LIMIT,
        ),
    },
])
display(validation_source_analysis)

          source   pairs  candidate_recall
0            ALS  300000           0.03250
1  Content-Based  100000           0.00675


### Дополнение source analysis

**Что делаем:** добавляем Personal History и Popularity  
**Зачем:** эти источники рассчитывались прямо в notebook  
**Что получим:** полную таблицу из четырёх строк

In [29]:
validation_source_analysis.loc[len(validation_source_analysis)] = {
    "source": "Personal History",
    "pairs": len(validation_personal_candidates),
    "candidate_recall": candidate_recall_at_k(
        validation_ground_truth_sample,
        validation_personal_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(),
        PERSONAL_LIMIT,
    ),
}
validation_source_analysis.loc[len(validation_source_analysis)] = {
    "source": "Popularity",
    "pairs": len(validation_popularity_candidates),
    "candidate_recall": candidate_recall_at_k(
        validation_ground_truth_sample,
        validation_popularity_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(),
        POPULARITY_LIMIT,
    ),
}
display(validation_source_analysis)

             source   pairs  candidate_recall
0               ALS  300000          0.032500
1     Content-Based  100000          0.006750
2  Personal History    6883          0.006292
3        Popularity   60000          0.015417


### Сохранение validation merge

**Что делаем:** записываем готовые candidates  
**Зачем:** notebook 08 не повторяет candidate generation  
**Что получим:** `merged_candidates_validation.parquet`

In [30]:
validation_merged_path = PROCESSED_DIR / "merged_candidates_validation.parquet"
validation_merged_candidates.to_parquet(validation_merged_path, index=False)
print("Сохранено:", validation_merged_path)

Сохранено: <PROJECT_ROOT>/data/processed/merged_candidates_validation.parquet


### Ground truth helper

**Что делаем:** фиксируем уже показанную подготовку ответов  
**Зачем:** функция не создаёт candidates  
**Что получим:** `build_ground_truth`

In [31]:
def build_ground_truth(target, known_users):
    target_evaluation = target[target["customer_id"].isin(known_users)]
    target_unique = (
        target_evaluation
        .sort_values("t_dat")
        .drop_duplicates(["customer_id", "article_id"])
    )
    return target_unique.groupby(
        "customer_id", sort=False
    )["article_id"].apply(list).to_dict()

### Sampling helper

**Что делаем:** фиксируем deterministic cohort  
**Зачем:** полный ground truth остаётся отдельным  
**Что получим:** users и sample dictionary

In [32]:
def sample_ground_truth(ground_truth, max_users, random_state):
    all_users = np.array(sorted(ground_truth))
    sample_size = min(max_users, len(all_users))
    rng = np.random.default_rng(random_state)
    users = rng.choice(all_users, size=sample_size, replace=False).tolist()
    sample = {customer_id: ground_truth[customer_id] for customer_id in users}
    return users, sample

### Personal History helper

**Что делаем:** фиксируем уже показанные recent и frequent шаги  
**Зачем:** функция не объединяет ALS, Content или Popularity  
**Что получим:** одну personal source table

In [33]:
def build_personal_candidates(history, customer_ids, limit):
    cohort = history[history["customer_id"].isin(customer_ids)].copy()
    recent = cohort.sort_values("t_dat", ascending=False).drop_duplicates(
        ["customer_id", "article_id"]
    )
    recent["personal_history_rank"] = recent.groupby("customer_id").cumcount() + 1
    recent = recent[recent["personal_history_rank"] <= limit].copy()
    recent["personal_history_score"] = 1 / recent["personal_history_rank"]
    frequent = cohort.groupby(["customer_id", "article_id"], as_index=False).agg(
        purchase_count=("article_id", "size"), last_purchase=("t_dat", "max")
    )
    frequent = frequent.sort_values(
        ["customer_id", "purchase_count", "last_purchase", "article_id"],
        ascending=[True, False, False, True],
    )
    frequent["personal_history_rank"] = frequent.groupby("customer_id").cumcount() + 1
    frequent = frequent[frequent["personal_history_rank"] <= limit].copy()
    frequent["personal_history_score"] = frequent["purchase_count"]
    columns = ["customer_id", "article_id", "personal_history_score", "personal_history_rank"]
    combined = pd.concat([recent[columns], frequent[columns]], ignore_index=True)
    return combined.groupby(["customer_id", "article_id"], as_index=False).agg(
        personal_history_score=("personal_history_score", "max"),
        personal_history_rank=("personal_history_rank", "min"),
    )

### Train: границы и данные

**Что делаем:** выбираем train-history и target  
**Зачем:** кандидаты должны оцениваться на своём temporal cutoff  
**Что получим:** `train_history` и `train_target`

In [34]:
train_cutoff = pd.Timestamp(windows["train"]["cutoff_date"])
train_end = pd.Timestamp(windows["train"]["target_end_date"])
train_history = transactions[
    transactions["t_dat"] < train_cutoff
].copy()
train_target = transactions[
    transactions["t_dat"].between(train_cutoff, train_end)
].copy()
assert train_history["t_dat"].max() < train_cutoff
print("History:", train_history.shape, "Target:", train_target.shape)

History: (998087, 5) Target: (13793, 5)


### Train: ground truth и cohort

**Что делаем:** повторяем уже показанную подготовку evaluation users  
**Зачем:** все candidate sources используют одинаковые ID  
**Что получим:** `train_users` и sample ground truth

In [35]:
train_ground_truth = build_ground_truth(
    train_target,
    set(train_history["customer_id"]),
)
train_users, train_ground_truth_sample = sample_ground_truth(
    train_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(train_users))

Evaluation users: 2000


### Train: ALS candidates

**Что делаем:** загружаем готовый Parquet и оставляем Top-150  
**Зачем:** ALS здесь не обучается  
**Что получим:** `train_als_candidates`

In [36]:
train_als_path = PROCESSED_DIR / "als_candidates_train.parquet"
train_als_candidates = pd.read_parquet(train_als_path)
train_als_candidates = train_als_candidates[
    train_als_candidates["als_rank"] <= ALS_LIMIT
].copy()
print("ALS rows:", len(train_als_candidates))

ALS rows: 300000


### Train: Content candidates

**Что делаем:** загружаем готовый Content-Based Parquet  
**Зачем:** OneHotEncoder и profiles здесь не создаются  
**Что получим:** `train_content_candidates`

In [37]:
train_content_path = PROCESSED_DIR / "content_candidates_train.parquet"
train_content_candidates = pd.read_parquet(train_content_path)
print("Content rows:", len(train_content_candidates))

Content rows: 100000


### Train: Personal History

**Что делаем:** повторяем уже разобранные recent и frequent операции  
**Зачем:** функция не объединяет остальные candidate sources  
**Что получим:** `train_personal_candidates`

In [38]:
train_personal_candidates = build_personal_candidates(
    train_history,
    train_users,
    PERSONAL_LIMIT,
)
print("Personal rows:", len(train_personal_candidates))

Personal rows: 7280


### Train: Popularity

**Что делаем:** строим небольшой глобальный список и повторяем его для cohort  
**Зачем:** операция быстрая и не использует target  
**Что получим:** `train_popularity_candidates`

In [39]:
train_popular_items = popular_items(
    train_history,
    limit=POPULARITY_LIMIT,
)
train_popularity_candidates = popularity_candidates(
    train_users,
    train_popular_items,
    limit=POPULARITY_LIMIT,
)
print("Popularity rows:", len(train_popularity_candidates))

Popularity rows: 60000


### Train: объединение источников

**Что делаем:** повторяем уже показанные concat/groupby/flags технической функцией  
**Зачем:** validation выше показывает реальную реализацию  
**Что получим:** `train_merged_candidates`

In [40]:
train_merged_candidates = merge_candidate_sources(
    als=train_als_candidates,
    content_based=train_content_candidates,
    personal_history=train_personal_candidates,
    popularity=train_popularity_candidates,
    limit_per_user=FINAL_LIMIT,
)
print("Merged rows:", len(train_merged_candidates))
print("Average candidates:", train_merged_candidates.groupby("customer_id").size().mean())

Merged rows: 445008
Average candidates: 222.504


### Train: Candidate Recall

**Что делаем:** сравниваем merged candidates с полным ground truth  
**Зачем:** оценка отделена от merge и сохранения  
**Что получим:** `train_candidate_recall`

In [41]:
train_candidate_lists = train_merged_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
train_candidate_recall = candidate_recall_at_k(
    train_ground_truth_sample,
    train_candidate_lists,
    FINAL_LIMIT,
)
print(f"Candidate Recall@{FINAL_LIMIT}:", train_candidate_recall)

Candidate Recall@250: 0.06933333333333333


### Train: сохранение

**Что делаем:** записываем одну строку на user-item pair  
**Зачем:** notebook 08 загрузит candidates без генерации с нуля  
**Что получим:** `merged_candidates_train.parquet`

In [42]:
train_merged_path = PROCESSED_DIR / "merged_candidates_train.parquet"
train_merged_candidates.to_parquet(train_merged_path, index=False)
print("Сохранено:", train_merged_path)

Сохранено: <PROJECT_ROOT>/data/processed/merged_candidates_train.parquet


### Test: границы и данные

**Что делаем:** выбираем test-history и target  
**Зачем:** кандидаты должны оцениваться на своём temporal cutoff  
**Что получим:** `test_history` и `test_target`

In [43]:
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
test_history = transactions[
    transactions["t_dat"] < test_cutoff
].copy()
test_target = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()
assert test_history["t_dat"].max() < test_cutoff
print("History:", test_history.shape, "Target:", test_target.shape)

History: (1035285, 5) Target: (13290, 5)


### Test: ground truth и cohort

**Что делаем:** повторяем уже показанную подготовку evaluation users  
**Зачем:** все candidate sources используют одинаковые ID  
**Что получим:** `test_users` и sample ground truth

In [44]:
test_ground_truth = build_ground_truth(
    test_target,
    set(test_history["customer_id"]),
)
test_users, test_ground_truth_sample = sample_ground_truth(
    test_ground_truth,
    MAX_EVALUATION_USERS,
    RANDOM_STATE,
)
print("Evaluation users:", len(test_users))

Evaluation users: 2000


### Test: ALS candidates

**Что делаем:** загружаем готовый Parquet и оставляем Top-150  
**Зачем:** ALS здесь не обучается  
**Что получим:** `test_als_candidates`

In [45]:
test_als_path = PROCESSED_DIR / "als_candidates_test.parquet"
test_als_candidates = pd.read_parquet(test_als_path)
test_als_candidates = test_als_candidates[
    test_als_candidates["als_rank"] <= ALS_LIMIT
].copy()
print("ALS rows:", len(test_als_candidates))

ALS rows: 300000


### Test: Content candidates

**Что делаем:** загружаем готовый Content-Based Parquet  
**Зачем:** OneHotEncoder и profiles здесь не создаются  
**Что получим:** `test_content_candidates`

In [46]:
test_content_path = PROCESSED_DIR / "content_candidates_test.parquet"
test_content_candidates = pd.read_parquet(test_content_path)
print("Content rows:", len(test_content_candidates))

Content rows: 100000


### Test: Personal History

**Что делаем:** повторяем уже разобранные recent и frequent операции  
**Зачем:** функция не объединяет остальные candidate sources  
**Что получим:** `test_personal_candidates`

In [47]:
test_personal_candidates = build_personal_candidates(
    test_history,
    test_users,
    PERSONAL_LIMIT,
)
print("Personal rows:", len(test_personal_candidates))

Personal rows: 7433


### Test: Popularity

**Что делаем:** строим небольшой глобальный список и повторяем его для cohort  
**Зачем:** операция быстрая и не использует target  
**Что получим:** `test_popularity_candidates`

In [48]:
test_popular_items = popular_items(
    test_history,
    limit=POPULARITY_LIMIT,
)
test_popularity_candidates = popularity_candidates(
    test_users,
    test_popular_items,
    limit=POPULARITY_LIMIT,
)
print("Popularity rows:", len(test_popularity_candidates))

Popularity rows: 60000


### Test: объединение источников

**Что делаем:** повторяем уже показанные concat/groupby/flags технической функцией  
**Зачем:** validation выше показывает реальную реализацию  
**Что получим:** `test_merged_candidates`

In [49]:
test_merged_candidates = merge_candidate_sources(
    als=test_als_candidates,
    content_based=test_content_candidates,
    personal_history=test_personal_candidates,
    popularity=test_popularity_candidates,
    limit_per_user=FINAL_LIMIT,
)
print("Merged rows:", len(test_merged_candidates))
print("Average candidates:", test_merged_candidates.groupby("customer_id").size().mean())

Merged rows: 445123
Average candidates: 222.5615


### Test: Candidate Recall

**Что делаем:** сравниваем merged candidates с полным ground truth  
**Зачем:** оценка отделена от merge и сохранения  
**Что получим:** `test_candidate_recall`

In [50]:
test_candidate_lists = test_merged_candidates.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
test_candidate_recall = candidate_recall_at_k(
    test_ground_truth_sample,
    test_candidate_lists,
    FINAL_LIMIT,
)
print(f"Candidate Recall@{FINAL_LIMIT}:", test_candidate_recall)

Candidate Recall@250: 0.05734999999999999


### Test: сохранение

**Что делаем:** записываем одну строку на user-item pair  
**Зачем:** notebook 08 загрузит candidates без генерации с нуля  
**Что получим:** `merged_candidates_test.parquet`

In [51]:
test_merged_path = PROCESSED_DIR / "merged_candidates_test.parquet"
test_merged_candidates.to_parquet(test_merged_path, index=False)
print("Сохранено:", test_merged_path)

Сохранено: <PROJECT_ROOT>/data/processed/merged_candidates_test.parquet


### Test alias

**Что делаем:** сохраняем совместимое имя test merge  
**Зачем:** следующие consumers могут использовать короткий путь  
**Что получим:** `merged_candidates.parquet`

In [52]:
merged_alias_path = PROCESSED_DIR / "merged_candidates.parquet"
test_merged_candidates.to_parquet(merged_alias_path, index=False)
print("Сохранено:", merged_alias_path)

Сохранено: <PROJECT_ROOT>/data/processed/merged_candidates.parquet


### Test source analysis

**Что делаем:** сохраняем coverage фактического final test  
**Зачем:** model comparison должен ссылаться на test, не validation  
**Что получим:** `candidate_source_analysis.csv`

In [53]:
test_source_analysis = pd.DataFrame([
    {"source": "ALS", "candidate_pairs": len(test_als_candidates),
     "candidate_recall": candidate_recall_at_k(
         test_ground_truth_sample,
         test_als_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(), ALS_LIMIT)},
    {"source": "Content-Based", "candidate_pairs": len(test_content_candidates),
     "candidate_recall": candidate_recall_at_k(
         test_ground_truth_sample,
         test_content_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(), CONTENT_LIMIT)},
    {"source": "Personal History", "candidate_pairs": len(test_personal_candidates),
     "candidate_recall": candidate_recall_at_k(
         test_ground_truth_sample,
         test_personal_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(), PERSONAL_LIMIT)},
    {"source": "Popularity", "candidate_pairs": len(test_popularity_candidates),
     "candidate_recall": candidate_recall_at_k(
         test_ground_truth_sample,
         test_popularity_candidates.groupby("customer_id")["article_id"].apply(list).to_dict(), POPULARITY_LIMIT)},
])
test_source_analysis.to_csv(REPORT_DIR / "candidate_source_analysis.csv", index=False)
display(test_source_analysis)

             source  candidate_pairs  candidate_recall
0               ALS           300000          0.033183
1     Content-Based           100000          0.003500
2  Personal History             7433          0.005750
3        Popularity            60000          0.022167


### Нормализация hybrid scores

**Что делаем:** масштабируем ALS, History и Popularity внутри пользователя  
**Зачем:** scores разных источников имеют разные диапазоны  
**Что получим:** три normalized columns

In [54]:
test_hybrid = test_merged_candidates.copy()

als_min = test_hybrid.groupby("customer_id")["als_score"].transform("min")
als_max = test_hybrid.groupby("customer_id")["als_score"].transform("max")
test_hybrid["normalized_als_score"] = (
    (test_hybrid["als_score"] - als_min) / (als_max - als_min).replace(0, np.nan)
).fillna(0)

history_min = test_hybrid.groupby("customer_id")["personal_history_score"].transform("min")
history_max = test_hybrid.groupby("customer_id")["personal_history_score"].transform("max")
test_hybrid["normalized_history_score"] = (
    (test_hybrid["personal_history_score"] - history_min)
    / (history_max - history_min).replace(0, np.nan)
).fillna(0)

### Popularity normalization

**Что делаем:** масштабируем последний ненормированный source  
**Зачем:** после этого фиксированная формула читается напрямую  
**Что получим:** `normalized_popularity_score`

In [55]:
popularity_min = test_hybrid.groupby("customer_id")["popularity_score"].transform("min")
popularity_max = test_hybrid.groupby("customer_id")["popularity_score"].transform("max")
test_hybrid["normalized_popularity_score"] = (
    (test_hybrid["popularity_score"] - popularity_min)
    / (popularity_max - popularity_min).replace(0, np.nan)
).fillna(0)

### Simple Hybrid score

**Что делаем:** применяем фиксированные веса 0.5/0.3/0.1/0.1  
**Зачем:** это понятный baseline перед CatBoost  
**Что получим:** `hybrid_score`

In [56]:
test_hybrid["hybrid_score"] = (
    0.5 * test_hybrid["normalized_als_score"]
    + 0.3 * test_hybrid["content_similarity_score"]
    + 0.1 * test_hybrid["normalized_history_score"]
    + 0.1 * test_hybrid["normalized_popularity_score"]
)
display(test_hybrid[["customer_id", "article_id", "hybrid_score"]].head())

                                         customer_id  article_id  hybrid_score
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0372860001      0.109330
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0562245046      0.330332
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0706016003      0.092308
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0759871002      0.128052
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0673396002      0.159916


### Simple Hybrid Top-12

**Что делаем:** сортируем score внутри пользователя  
**Зачем:** получаем standalone рекомендации без CatBoost  
**Что получим:** `hybrid_recommendations`

In [57]:
hybrid_top12 = test_hybrid.sort_values(
    ["customer_id", "hybrid_score", "article_id"],
    ascending=[True, False, True],
).groupby("customer_id", sort=False).head(12)
hybrid_recommendations = hybrid_top12.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
display(hybrid_top12.head())

                                          customer_id  ... hybrid_score
14  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...     0.500000
17  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...     0.411047
22  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...     0.349338
23  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...     0.343743
1   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...     0.330332

[5 rows x 19 columns]


### Simple Hybrid metrics

**Что делаем:** оцениваем test Top-12 и сохраняем CSV  
**Зачем:** notebook 10 сравнит baseline с CatBoost  
**Что получим:** `simple_hybrid_metrics.csv`

In [58]:
hybrid_metrics = {
    "model": "Simple Hybrid",
    "Recall@12": mean_recall_at_k(test_ground_truth_sample, hybrid_recommendations, 12),
    "MAP@12": map_at_k(test_ground_truth_sample, hybrid_recommendations, 12),
    "HitRate@12": hit_rate_at_k(test_ground_truth_sample, hybrid_recommendations, 12),
    "Candidate Recall": test_candidate_recall,
    "users_evaluated": len(test_ground_truth_sample),
    "average_candidates": test_merged_candidates.groupby("customer_id").size().mean(),
    "notes": "Fixed 0.5/0.3/0.1/0.1 weights",
}
pd.DataFrame([hybrid_metrics]).to_csv(
    REPORT_DIR / "simple_hybrid_metrics.csv", index=False
)
display(pd.Series(hybrid_metrics))

model                                 Simple Hybrid
Recall@12                                  0.003167
MAP@12                                     0.001369
HitRate@12                                   0.0035
Candidate Recall                            0.05735
users_evaluated                                2000
average_candidates                         222.5615
notes                 Fixed 0.5/0.3/0.1/0.1 weights
dtype: object
